# Лабораторная работа 10. Фильтрация и сглаживание сигналов

**Цель.** Выполнить частотную фильтрацию аудиосигнала и сравнить варианты экспоненциального сглаживания.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.fft import rfft, rfftfreq, irfft

np.random.seed(42)
audio_candidates = [Path('Wav_26mb.wav'), Path('data/Wav_26mb.wav')]
audio_path = next((path for path in audio_candidates if path.exists()), audio_candidates[0])


## Подготовка входных данных

В этом блоке считываются данные, с которыми дальше будет работать лабораторная. После загрузки важно проверить формат индекса, названия колонок и размерность ряда, потому что от этого зависит корректность всех следующих расчётов.


In [ ]:
if not audio_path.exists():
    print('Аудиофайл не найден, блок фильтрации пропущен.')
else:
    samplerate, data = wavfile.read(audio_path)
    if data.ndim > 1:
        data = data[:, 0]
    time = np.arange(len(data)) / samplerate
    spectrum = rfft(data)
    freqs = rfftfreq(len(data), 1 / samplerate)
    filtered = spectrum.copy()
    filtered[freqs > 2000] = 0
    restored = irfft(filtered, n=len(data))

    print('Sampling Rate:', samplerate)
    print('Audio Shape:', data.shape)
    plt.figure(figsize=(12, 5))
    plt.plot(time[:5000], data[:5000], label='исходный сигнал', alpha=0.7)
    plt.plot(time[:5000], restored[:5000], label='после low-pass FFT', alpha=0.7)
    plt.legend()
    plt.xlabel('Время, с')
    plt.ylabel('Амплитуда')
    plt.show()


## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# Демонстрация экспоненциального сглаживания на воспроизводимом синтетическом ряде.
t = np.arange(120)
series = pd.Series(0.05 * t + np.sin(t / 6) + np.random.normal(scale=0.25, size=len(t)))
smoothed = series.ewm(alpha=0.25, adjust=False).mean()

plt.figure(figsize=(10, 4))
plt.plot(series, label='исходный ряд')
plt.plot(smoothed, label='экспоненциальное сглаживание')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Вывод

FFT-фильтрация убирает частоты выше выбранного порога при наличии аудиофайла. Экспоненциальное сглаживание демонстрирует, как можно уменьшить шум без явного перехода в частотную область.
